# FinBERT Benchmark Orchestration

This notebook stays library-driven. It can optionally construct the benchmark datasets first if the manifest you want to benchmark does not exist yet.

In [ ]:
from pathlib import Path

from thesis_pkg.benchmarking import BenchmarkSampleSpec
from thesis_pkg.benchmarking import BucketBatchConfig
from thesis_pkg.benchmarking import FinbertBenchmarkRunConfig
from thesis_pkg.benchmarking import FinbertBenchmarkSuiteConfig
from thesis_pkg.benchmarking import FinbertBenchmarkSweepConfig
from thesis_pkg.benchmarking import FinbertRuntimeConfig
from thesis_pkg.benchmarking import SentenceDatasetConfig
from thesis_pkg.benchmarking import build_finbert_benchmark_suite
from thesis_pkg.benchmarking import run_finbert_benchmark
from thesis_pkg.benchmarking import run_finbert_benchmark_sweep

In [ ]:
BENCHMARK_SAMPLE_NAME = "5pct"  # Change to "1pct" to benchmark the smaller sample.

BENCHMARK_DATASET_OUT_ROOT = Path("/content/drive/MyDrive/Data_LM/results/benchmarking/finbert_inference")
DATASET_MANIFEST_PATH = (
    BENCHMARK_DATASET_OUT_ROOT
    / f"finbert_10k_items_{BENCHMARK_SAMPLE_NAME}_seed42"
    / "benchmark_manifest.json"
)
RUNS_OUT_ROOT = Path("/content/drive/MyDrive/Data_LM/results/benchmarking/finbert_runtime_runs")

# Inferred from sec_ccm_unified_runner.py: RUN_ROOT / "items_analysis" for the Drive profile.
FULL_ITEMS_ANALYSIS_DIR = Path("/content/drive/MyDrive/Data_LM/results/sec_ccm_unified_runner/items_analysis")

BUILD_DATASET_IF_MISSING = False
ENABLE_SENTENCE_DATASET_BUILD = False
WRITE_FULL_UNIVERSE_TOKEN_AUDIT = False
BENCHMARK_BUILD_SEED = 42

DATASET_SAMPLE_SPECS = (
    BenchmarkSampleSpec(sample_name="1pct", sample_fraction=0.01),
    BenchmarkSampleSpec(sample_name="5pct", sample_fraction=0.05),
)

print(f"Benchmark sample selected: {BENCHMARK_SAMPLE_NAME}")
print(f"Benchmark manifest path: {DATASET_MANIFEST_PATH}")

In [ ]:
if BUILD_DATASET_IF_MISSING and not DATASET_MANIFEST_PATH.exists():
    dataset_artifacts = build_finbert_benchmark_suite(
        FinbertBenchmarkSuiteConfig(
            source_items_dir=FULL_ITEMS_ANALYSIS_DIR,
            out_root=BENCHMARK_DATASET_OUT_ROOT,
            sample_specs=DATASET_SAMPLE_SPECS,
            seed=BENCHMARK_BUILD_SEED,
            write_full_universe_token_audit=WRITE_FULL_UNIVERSE_TOKEN_AUDIT,
            sentence_dataset=SentenceDatasetConfig(enabled=ENABLE_SENTENCE_DATASET_BUILD),
        )
    )
    DATASET_MANIFEST_PATH = dataset_artifacts[BENCHMARK_SAMPLE_NAME].manifest_path
    print(f"Built benchmark datasets under: {BENCHMARK_DATASET_OUT_ROOT}")
    print(f"Using manifest for {BENCHMARK_SAMPLE_NAME}: {DATASET_MANIFEST_PATH}")
elif DATASET_MANIFEST_PATH.exists():
    print(f"Using existing manifest for {BENCHMARK_SAMPLE_NAME}: {DATASET_MANIFEST_PATH}")
else:
    print(f"Manifest for {BENCHMARK_SAMPLE_NAME} does not exist yet.")
    print(f"Set BUILD_DATASET_IF_MISSING = True to build from: {FULL_ITEMS_ANALYSIS_DIR}")

In [ ]:
single_run_cfg = FinbertBenchmarkRunConfig(
    dataset_manifest_path=DATASET_MANIFEST_PATH,
    out_root=RUNS_OUT_ROOT,
    batch_config=BucketBatchConfig(
        name="baseline",
        short_batch_size=64,
        medium_batch_size=32,
        long_batch_size=16,
    ),
    run_name=f"{BENCHMARK_SAMPLE_NAME}_baseline_run",
)

sweep_cfg = FinbertBenchmarkSweepConfig(
    base_run=single_run_cfg,
    batch_configs=(
        BucketBatchConfig(name="small", short_batch_size=32, medium_batch_size=16, long_batch_size=8),
        BucketBatchConfig(name="baseline", short_batch_size=64, medium_batch_size=32, long_batch_size=16),
        BucketBatchConfig(name="large", short_batch_size=128, medium_batch_size=64, long_batch_size=32),
    ),
)

runtime_cfg = FinbertRuntimeConfig(device=None, use_autocast=True, amp_dtype="auto")

In [ ]:
single_run_artifacts = run_finbert_benchmark(single_run_cfg, runtime=runtime_cfg)
single_run_artifacts

In [ ]:
sweep_summary_df = run_finbert_benchmark_sweep(sweep_cfg, runtime=runtime_cfg)
sweep_summary_df